# Sample Generation (unconditional)

Generates N samples per trained model using the chosen diffusers sampler
(DDIM, DDPM, DPM-Solver++, or UniPC). Output: per-model folder with N
`.nii.gz` volumes and a `stats.json` with timing and voxel statistics.

Supports both 32^3 and 48^3 resolutions, pick `IMAGE_SIZE` below.

- `IMAGE_SIZE = 32` -> all five 32³ experiments (baseline, data_augmentation,
  linear_schedule, no_attention, noise_prediction)
- `IMAGE_SIZE = 48` -> baseline only (the single 48^3 experiment we trained)

To switch samplers, change the `SAMPLER` knob and rerun.


In [ ]:
from pathlib import Path

IMAGE_SIZE     = 32         # 32 | 48
SAMPLER        = "ddim"     # "ddim" | "ddpm" | "dpm_solver++" | "unipc"
N_SAMPLES      = 1024
SAMPLER_STEPS  = 50
SEED_BASE      = 42

CHECKPOINT_ROOT = Path(rf"C:\Users\Sven\Desktop\Data\diffusion\{IMAGE_SIZE}")
OUTPUT_ROOT     = Path(rf"samples/{IMAGE_SIZE}/{SAMPLER.upper()}")

print(f"size={IMAGE_SIZE}  sampler={SAMPLER}  steps={SAMPLER_STEPS}  N={N_SAMPLES}")
print(f"checkpoints: {CHECKPOINT_ROOT}")
print(f"output:      {OUTPUT_ROOT}")


In [ ]:
import sys
import time
import json
from dataclasses import dataclass
from typing import Tuple

import numpy as np
import nibabel as nib
import torch
from tqdm.auto import tqdm

SRC = Path.cwd() / "src"
sys.path.insert(0, str(SRC))

from model.unet3d import UNet3D
from sampler.sampler import make_scheduler, sample


# Pickled inside the training checkpoints — needs to be importable by
# torch.load(weights_only=False), which means it must live in __main__.
@dataclass
class Config:
    data_path: str = ""
    output_path: str = ""
    image_size: int = 32
    in_channels: int = 1
    model_channels: int = 96
    channel_mult: Tuple[int, ...] = (1, 2, 4, 4)
    num_res_blocks: int = 2
    attention_resolutions: Tuple[int, ...] = (8, 4)
    num_groups: int = 8
    dropout: float = 0.0
    num_timesteps: int = 250
    beta_schedule: str = "cosine"
    prediction_type: str = "v"
    num_epochs: int = 300
    batch_size: int = 10
    learning_rate: float = 2e-4
    warmup_steps: int = 500
    ema_decay: float = 0.9999
    grad_clip: float = 1.0
    use_loss_aware_sampling: bool = True
    loss_history_size: int = 10
    snr_gamma: float = 5.0
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


sys.modules.setdefault("__main__", sys.modules[__name__])
setattr(sys.modules["__main__"], "Config", Config)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")


In [ ]:
# Attention is applied at resolutions matching the trained model. The 32³ models
# used (8, 4); the 48³ baseline used (12, 6).
ATTN_BY_SIZE = {32: (8, 4), 48: (12, 6)}

MODELS_BY_SIZE = {
    32: {
        "baseline":          {"output_dir": "output_baseline",          "prediction_type": "v",       "beta_schedule": "cosine", "attention_resolutions": ATTN_BY_SIZE[32]},
        "data_augmentation": {"output_dir": "output_data_augmentation", "prediction_type": "v",       "beta_schedule": "cosine", "attention_resolutions": ATTN_BY_SIZE[32]},
        "linear_schedule":   {"output_dir": "output_linear_schedule",   "prediction_type": "v",       "beta_schedule": "linear", "attention_resolutions": ATTN_BY_SIZE[32]},
        "no_attention":      {"output_dir": "output_no_attention",      "prediction_type": "v",       "beta_schedule": "cosine", "attention_resolutions": ()},
        "noise_prediction":  {"output_dir": "output_noise_prediction",  "prediction_type": "epsilon", "beta_schedule": "cosine", "attention_resolutions": ATTN_BY_SIZE[32]},
    },
    48: {
        "baseline_48":       {"output_dir": "output_baseline_48",       "prediction_type": "v",       "beta_schedule": "cosine", "attention_resolutions": ATTN_BY_SIZE[48]},
    },
}

assert IMAGE_SIZE in MODELS_BY_SIZE, f"no model registry for IMAGE_SIZE={IMAGE_SIZE}"
MODELS = MODELS_BY_SIZE[IMAGE_SIZE]
print(f"{IMAGE_SIZE}³ — {len(MODELS)} model(s): {list(MODELS)}")


In [ ]:
def load_model(spec):
    cfg = Config(image_size=IMAGE_SIZE, attention_resolutions=spec["attention_resolutions"])
    model = UNet3D(cfg).to(device)

    ckpt_path = CHECKPOINT_ROOT / spec["output_dir"] / "checkpoints" / "final_model.pt"
    ckpt = torch.load(str(ckpt_path), map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])

    # apply the EMA shadow weights (these are what we trained on, not the raw params)
    if "ema" in ckpt and "shadow" in ckpt["ema"]:
        for name, p in model.named_parameters():
            if name in ckpt["ema"]["shadow"]:
                p.data.copy_(ckpt["ema"]["shadow"][name].to(device))

    model.eval()
    return model, ckpt_path


In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
summary = []

for model_name, spec in MODELS.items():
    out_dir = OUTPUT_ROOT / f"samples_{model_name}"
    (out_dir / "samples").mkdir(parents=True, exist_ok=True)

    print("=" * 60)
    print(f"{model_name}: pred={spec['prediction_type']}  "
          f"sched={spec['beta_schedule']}  attn={spec['attention_resolutions']}")

    model, ckpt_path = load_model(spec)
    scheduler = make_scheduler(SAMPLER, spec["beta_schedule"], spec["prediction_type"])

    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()

    per_sample = []
    for i in tqdm(range(N_SAMPLES), desc=f"gen {model_name}"):
        seed = SEED_BASE + i
        t0 = time.time()
        vol = sample(model, scheduler, IMAGE_SIZE,
                     num_steps=SAMPLER_STEPS, device=device, seed=seed)
        dt = time.time() - t0

        arr = vol.squeeze().cpu().numpy().astype(np.float32)
        out_path = out_dir / "samples" / f"sample_{i+1:04d}.nii.gz"
        nib.save(nib.Nifti1Image(arr, np.eye(4)), str(out_path))

        per_sample.append({
            "filename": out_path.name,
            "seed": seed,
            "time_sec": round(dt, 3),
            "voxel_min":  round(float(arr.min()), 4),
            "voxel_max":  round(float(arr.max()), 4),
            "voxel_mean": round(float(arr.mean()), 4),
            "voxel_std":  round(float(arr.std()), 4),
        })

    peak_gb = round(torch.cuda.max_memory_allocated() / 1e9, 2) if device == "cuda" else None
    with (out_dir / "stats.json").open("w") as f:
        json.dump({
            "model": model_name,
            "image_size": IMAGE_SIZE,
            "sampler": SAMPLER,
            "steps": SAMPLER_STEPS,
            "n_samples": N_SAMPLES,
            "checkpoint": str(ckpt_path),
            "peak_gpu_gb": peak_gb,
            "per_sample": per_sample,
        }, f, indent=2)

    summary.append({"model": model_name, "n": len(per_sample), "peak_gb": peak_gb})
    del model
    if device == "cuda":
        torch.cuda.empty_cache()

print("\nsummary:")
for s in summary:
    print(s)
